# AB-EMCTS on the R7615 — run everything from here

Companion to `analysis/AB_EMCTS_R7615_PLAN.md` (what each stage measures and why) and
`analysis/AB_EMCTS_RUNBOOK.md` (what to check before continuing). This notebook is the thing that
runs them.

**The operating loop.** The repo is cloned into Jupyter; stages run here; results land under
`~/ab_runs`, **outside the clone**, so merging a PR and re-cloning never touches them. Every stage
ends with a `bundle(...)` cell that zips the pre-specified results into `~/ab_runs/to_claude/` —
download that one file and send it back for analysis. Do not rearrange this: the portfolio is pinned
(a rebuild silently re-pairs every visit seed) and the runners resume from checkpoints in that tree.

**Order matters.** Stage A gates everything. Stage B comes before D — it pins the selection substrate
that D, E, F and G are all measured on. Every cell is independently re-runnable: runners resume from
their checkpoints, and the setup steps skip themselves once their outputs exist.

## Parameters and helpers

Run this first, always — everything below uses it. It finds `Tribes-master/` on its own (a Jupyter
kernel's working directory is the *notebook's* folder, not where the server was launched, so nothing
here depends on it), and it routes every JVM at the calibrated weights via `-Dcompute.weightsFile` —
without that flag the meter would silently read the v6 tree's weights and `requireProfiles` would
abort every run.

`WORKERS` and `THREADS_400K` are Stage A0b's measured answers (2026-08-28): 54 workers at the
cheap levels, but only 31 at 400k, where per-task heap makes 64 GB the binding constraint — the
runner takes `-Dstage1.threads.400000` so one level's ceiling does not drag the others down.
`MACHINE_FACTOR` only scales the banner estimates (1.5 fitted from A0b; Stage B checks it). After Gate D, set
`CHAMPION` and flip `CHAMPION_CONFIRMED` here, then re-run this cell — Stages E and F refuse to start
on the unconfirmed placeholder.

In [ ]:
import os, sys, subprocess, pathlib, datetime, shutil, zipfile, glob

TRIBES = ""     # leave empty: auto-detected. Set to an absolute path only if the banner says so.

def _find_root():
    if TRIBES:
        return pathlib.Path(TRIBES).resolve()
    here = pathlib.Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / "lib").is_dir() and (base / "src").is_dir():
            return base
        tm = base / "Tribes-master"
        if (tm / "lib").is_dir() and (tm / "src").is_dir():
            return tm
    raise RuntimeError(
        f"Cannot find Tribes-master at or above {here}. A Jupyter kernel's working directory is the "
        "notebook's own folder (and it keeps the OLD folder if the file was moved without a kernel "
        "restart) - the walk-up handles the usual layouts, so if it failed, either restart the "
        "kernel or the clone is incomplete (no lib/ or src/). Last resort: set TRIBES at the top of "
        "this cell to the absolute path of Tribes-master and re-run.")

ROOT = _find_root()
os.chdir(ROOT)

# ---- where everything lives: OUTSIDE the clone, so a merge + re-clone never touches it ----
DATA        = pathlib.Path.home() / "ab_runs"
BASE        = DATA / "stage1"           # isolation-phase tree (portfolio, calibration, dev, holdout)
PORT        = BASE / "_portfolio"
CALIB       = BASE / "_calibration"
STAGE0_BASE = DATA / "stage0"           # Stage C's logged games
RESULTS     = DATA / "ab_results"       # analysis tables, figures, SUMMARY.md per stage
LOGS        = DATA / "logs"             # every run's console output, teed
TO_CLAUDE   = DATA / "to_claude"        # one zip per stage: download and send back
for _d in (DATA, LOGS, TO_CLAUDE):
    _d.mkdir(parents=True, exist_ok=True)

# ---- machine ----
WORKERS        = 54                     # A0b measured: 47 @ 40k, 54 @ 80k (flat past 39)
THREADS_400K   = 31                     # A0b: the 400k level is heap-bound at -Xmx64g (peak 65 GB
                                        # at 31 workers). Raise this ONLY together with HEAP, and
                                        # only after `free -g` (A0 cell) shows the RAM for it.
HEAP           = "64g"                  # run JVMs; 400k rides this cap at THREADS_400K workers
CALIB_HEAP     = "32g"                  # calibration JVM
GC_FLAGS       = "-XX:+UseG1GC -XX:MaxGCPauseMillis=500"   # add -XX:+UseNUMA if A0 reports >1 node
MACHINE_FACTOR = "1.5"                  # fitted from A0b throughput at the chosen W; Stage B checks it
LEVELS         = "40000,80000,400000"

# ---- Stage D's outcome, confirmed by hand after Gate D ----
CHAMPION           = "Short5_e25"       # <- Stage D's winner, before running E and F
CHAMPION_CONFIRMED = False              # <- set True once Gate D is read and CHAMPION is set

WEIGHTS = CALIB / "compute_weights.properties"   # every JVM below is pointed here
CP      = os.pathsep.join(["out", "lib/*"])      # ':' on Linux, ';' on Windows
PY      = sys.executable                         # the interpreter running this kernel


def run(cmd, log=None, check=False):
    """Run a command, stream it here, and tee it to <DATA>/logs/<log>.log (append).

    Teeing matters: these stages run for hours, and a dropped connection or a closed browser tab
    would otherwise take the only copy of the output with it. Logs append across re-runs, so a
    resumed stage keeps its earlier half too.
    """
    path = LOGS / f"{log or 'cell'}.log"
    started = datetime.datetime.now()
    print(f"$ {cmd}\n  -> {path}\n", flush=True)
    with open(path, "a", encoding="utf-8") as fh:
        fh.write(f"\n>>>>> {started:%Y-%m-%d %H:%M:%S} $ {cmd}\n")
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, encoding="utf-8", errors="replace")
        for line in p.stdout:
            print(line, end="", flush=True)
            fh.write(line)
        rc = p.wait()
        fh.write(f"===== exit {rc} after {datetime.datetime.now() - started}\n")
    print(f"\n[exit {rc}] after {datetime.datetime.now() - started}", flush=True)
    if check and rc != 0:
        raise RuntimeError(f"exit {rc}: {cmd}\n  full output: {path}")
    return rc


def java(main, args="", heap=None, props="", log=None, check=False):
    """Every JVM gets the weights pointer: without it, ComputeMeter defaults to the v6 tree."""
    cmd = (f'java -Xmx{heap or HEAP} {GC_FLAGS} -Dcompute.weightsFile="{WEIGHTS}" {props} '
           f'-cp "{CP}" {main} {args}')
    return run(cmd, log=log or main.split(".")[-1], check=check)


def latest_log(log):
    """The output of the most recent run appended to <log>.log."""
    txt = (LOGS / f"{log}.log").read_text(encoding="utf-8", errors="replace")
    return txt.rsplit("\n>>>>> ", 1)[-1]


def assert_in_log(log, needle):
    assert needle in latest_log(log), (
        f"'{needle}' not found in the latest {log} run - read {LOGS / (log + '.log')} before continuing")
    print(f"[gate] '{needle}' confirmed in {log}")


def bundle(name, items, note=""):
    """Zip the pre-specified per-stage results into to_claude/<name>.zip - download and send back.

    items may be files, directories (recursive) or glob patterns. Heavy raw streams
    (seqlog_*.csv.gz, mutlog_*.csv.gz) and lock files stay on the server - ask for specific ones by
    name if a question needs them.
    """
    EXCLUDE = ("seqlog_", "mutlog_", "_running.lock")
    out = TO_CLAUDE / f"{name}.zip"
    if out.exists():
        out.unlink()
    files = []
    for it in items:
        p = pathlib.Path(it)
        if not p.is_absolute():
            p = DATA / p
        if p.is_dir():
            files += [q for q in p.rglob("*") if q.is_file()]
        elif p.exists():
            files.append(p)
        else:
            hits = [pathlib.Path(h) for h in glob.glob(str(p)) if pathlib.Path(h).is_file()]
            files += hits
            if not hits:
                print(f"  (missing, skipped: {p})")
    files = sorted({f for f in files if not any(x in f.name for x in EXCLUDE)})
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for f in files:
            z.write(f, str(f.relative_to(DATA)) if DATA in f.parents else f.name)
    print(f"\n{len(files)} files, {out.stat().st_size / 1e6:.1f} MB -> {out}")
    print(f"DOWNLOAD AND SEND TO CLAUDE: {out}")
    if note:
        print(note)
    return out


try:
    import pandas, numpy
    _py_note = f"pandas {pandas.__version__}, numpy {numpy.__version__}"
except ImportError as _e:
    _py_note = f"MISSING {_e} - the analyze cells will fail; install pandas+numpy into this kernel"
print("Tribes root :", ROOT)
print("data root   :", DATA, "(outside the clone - survives re-clones)")
print("weights     :", WEIGHTS)
print("python      :", PY, "|", _py_note)
print("workers     :", WORKERS, "| levels:", LEVELS)
print("champion    :", CHAMPION, "(CONFIRMED)" if CHAMPION_CONFIRMED else "(placeholder - confirm after Gate D)")

---
## 0. Build

`--release 11` pins the language level *and* the API set to the server's JDK, so anything newer is a
compile error here rather than a surprise mid-run. The source list is regenerated in Python each
time, so new classes are picked up automatically and nothing depends on shell tools.

In [ ]:
srcs = sorted(str(p) for p in pathlib.Path("src").rglob("*.java"))
pathlib.Path("sources.txt").write_text("\n".join(srcs) + "\n", encoding="utf-8")
print(f"{len(srcs)} source files")
shutil.rmtree("out", ignore_errors=True)
run('javac --release 11 -nowarn -d out -cp "lib/*" @sources.txt', log="build", check=True)
print("BUILD OK -", sum(1 for _ in pathlib.Path("out").rglob("*.class")), "classes")

### The shakedown — ~6 min, 18 checks

Stage A is a ~5.6 h block that is mostly single-threaded and needs an idle machine, so an environment
problem found inside it costs an evening. This exercises every moving part at toy size. It measures
nothing.

**Read two things by eye even on a pass:** cores must be **detected**, not `ASSUMED`; and the working
tree must be on local disk, because the calibration is a timing measurement. (The shakedown's own
scratch output goes to `test_results/_shakedown` inside the clone — that is deliberate, it is
disposable.)

In [ ]:
run("bash run_ab_shakedown.sh", log="shakedown", check=True)
bundle("shakedown", [LOGS / "shakedown.log"])

---
## Stage A — machine, currency, gates  *(~5.6 h; nothing measured)*

### A0 — what this machine will do

In [ ]:
java("core.data.exp.ServerProfile", heap="4g", log="A0_profile", check=True)
run("free -g", log="A0_profile")   # RAM decides whether the 400k level may ever run wider

### A1 — the portfolio and the frozen split *(~0.3 h, runs once, then pins itself)*

400 scenarios (200 single + 200 dual), built **once** for the whole line: `visitSeed` keys on
scenario id, so a rebuild silently re-pairs everything. The cell skips itself once the manifest and
both split files exist.

Read the freeze report before trusting the split — the halves should differ by at most one scenario
and a couple of branching points, not by twenty.

In [ ]:
if (PORT / "portfolio.csv").exists() and (PORT / "_split_dev.txt").exists():
    print("portfolio built and split frozen already - PINNED, skipping.")
    print(f"(deleting {PORT} restarts the entire line; never do it between stages)")
else:
    java("core.data.exp.BuildV6Portfolio", f'"{PORT}" {WORKERS} 200 200',
         log="A1_portfolio", check=True)
    java("core.data.exp.PortfolioSplit", f'"{PORT}"', heap="4g", log="A1_split", check=True)

### A0b — thread calibration, per budget level *(needs the portfolio, hence after A1)*

Tree size grows with the budget, so the best thread count at 400k is not the one at 40k. Record `W`
per level, and read `ThrottlePct` / `ClockDropPct`: a point that only reaches peak on borrowed boost
headroom is not the one to run for hours.

**Set `WORKERS` in the parameters cell from this, then re-run that cell.**

In [ ]:
if (CALIB / "thread_calibration.csv").exists():
    print(f"already measured - delete {CALIB / 'thread_calibration.csv'} to redo.")
else:
    java("core.data.exp.ThreadCalibrationV6", f'"{PORT}" "{CALIB}" {LEVELS}', log="A0_threads",
         check=True)

### A2 — calibrate idle *(~2.6 h, single-threaded)*

54 cells: the reference `EMCTS` plus all 17 action-bias arms, × 3 levels.

**Nothing else may run on this machine while this does.** `auto` keeps the suite's own turn counts
(`8,4,2`) — do not replace it with a number unless you mean to, because the zero-eval arms make ~4×
fewer copies by construction and need the extra turns at 40k to clear the sample floor. Verdict must
read `PASS`; on `FAIL` the weights are deliberately not written and the measurements are still in
`compute_profiles.csv`. The cell skips itself once the weights file exists.

In [ ]:
if WEIGHTS.exists():
    print(f"weights already written - delete {WEIGHTS} to recalibrate (then re-run the floor probe too).")
else:
    java("core.data.exp.ComputeCalibrateV6", f'"{PORT}" "{CALIB}" 3 auto 6 {LEVELS} --suite ab',
         heap=CALIB_HEAP, props="-XX:+AlwaysPreTouch", log="A2_calibrate", check=True)

### A3′ — calibrate under load — **RESOLVED 2026-08-29, do not re-run**

The 2026-08-28 run answered this, despite reporting FAIL. The harness *did* load the machine —
every op cost 42–78% more than idle — but the OS load average is meaningless inside this container
(0.11/core while saturated), so the old gate discarded a measurement that had succeeded (it now
gates on the harness's own completed work instead). The measured answer, recorded in the plan:
non-Z0 ratios hold within ~6% under load; **`Z0_Fine` moves ~+24%** at 80k and 400k — the
cache-light arm loses its idle cache advantage to L3 contention, so idle pricing *subsidizes* the
`Z0_*` family at run conditions.

**Resolution (amended from the pre-registered rule, deliberately):** the idle 54-cell calibration
stays the one currency — a loaded currency would be specific to one `W` on one container — and
**W7 wall-clock fairness is CO-PRIMARY everywhere**: a units-only win, `Z0_*` above all, is
unresolved until it survives W7 (now built into `analyze_stage1.py`).

In [ ]:
RERUN_A3 = False   # resolved 2026-08-29; flip only on a deliberate decision to re-measure
if not RERUN_A3:
    print("A3' resolved by amendment - idle currency pinned, W7 co-primary; see the plan's")
    print("Gate A3' resolution. The 2026-08-28 measurements are archived in stageA.zip.")
elif list(CALIB.glob("compute_profiles_load*.csv")):
    print("loaded calibration already recorded - delete its compute_*_load* files to redo.")
else:
    java("core.data.exp.ComputeCalibrateV6",
         f'"{PORT}" "{CALIB}" 3 auto 6 {LEVELS} --suite a3 --underLoad {WORKERS}',
         heap=CALIB_HEAP, props="-XX:+AlwaysPreTouch", log="A3_loaded", check=True)

### A4 — the gates

Three questions: is Baseline still bit-identical to plain EMCTS, is the Stage-0 instrumentation still
inert, and **does the bias actually have evidence to act on at each budget**.

That last one is the floor probe. The action-bias line fails quietly from *below*: `qhat` shrinks
toward the running global mean, so when every candidate key is unseen they all tie and the tie-break
is uniform — the arm becomes Baseline with extra bookkeeping and still reports a small noisy effect
rather than announcing that the mechanism never fired. It is weight-dependent, which is why it runs
**after** A2 and must be re-taken whenever the weights change.

In [ ]:
java("core.data.exp.Stage1Runner", "verify", heap="8g", log="A4_verify_stage1", check=True)
assert_in_log("A4_verify_stage1", "PASS")
java("core.data.exp.Stage0SignalRunner", "verify 40000 20", heap="8g", log="A4_verify_stage0", check=True)
assert_in_log("A4_verify_stage0", "PASS")
java("core.data.exp.ActionBiasFloorProbe", f'"{PORT}" "{CALIB}" {LEVELS} 20 {WORKERS}',
     log="A4_floor", check=True)

> ### Gate A
> Both verifies PASS &middot; A2 verdict PASS &middot; A3′ verdict recorded &middot; floor probe
> passes at 80k &middot; split frozen and balanced &middot; cores detected.
> **Nothing downstream runs otherwise.**

A level that fails only the floor probe is a *reporting* level — it can carry a budget trend or an
anytime curve, but not a claim that the bias helped, because at that budget the bias was not
operating.

**Send `to_claude/stageA.zip` before running Stage B** — the calibration verdicts, the load
comparison, the floor probe and the split report all get read on the other end before the machine
commits to the measured stages.

In [ ]:
bundle("stageA", [
    LOGS / "A0_profile.log", LOGS / "A0_threads.log",
    LOGS / "A1_portfolio.log", LOGS / "A1_split.log",
    LOGS / "A2_calibrate.log", LOGS / "A3_loaded.log",
    LOGS / "A4_verify_stage1.log", LOGS / "A4_verify_stage0.log", LOGS / "A4_floor.log",
    CALIB / "compute_profiles.csv", CALIB / "compute_weights.properties",
    str(CALIB / "compute_profiles_load*.csv"), str(CALIB / "compute_weights_load*.properties"),
    CALIB / "thread_calibration.csv", CALIB / "action_bias_floor.csv",
    PORT / "portfolio.csv", PORT / "_split_dev.txt", PORT / "_split_holdout.txt",
])

---
## Stage B — the substrate  *(~0.4 h)* — run this before D

Four arms at 80k on the development half: `Baseline`, `Bias_e25`, `Comma_Off`, `Comma_On`. The two
plus arms are the real `Baseline` and `Bias_e25` rather than renamed copies, so Stage D later
**skips them on resume** — same checkpoint keys, no work done twice.

In [ ]:
java("core.data.exp.Stage1Runner", f'{WORKERS} "{BASE}" dev B',
     props=f"-Dstage1.machineFactor={MACHINE_FACTOR}", log="B", check=True)
run(f'"{PY}" analysis/analyze_stage1.py "{BASE / "dev"}" --outdir "{RESULTS / "B"}"',
    log="B_analysis", check=True)

In [ ]:
print((RESULTS / "B" / "SUMMARY.md").read_text(encoding="utf-8"))

> ### Gate B — read two things, and stop until you have them
> 1. **The main effect** `Comma_Off − Plus_Off`. Whichever wins becomes the substrate *and the
>    baseline* for D–G. If comma wins, that is a re-pin, not a footnote.
> 2. **The design section** — the measured ICC, which sets Stage D's visit count. The default is 8.
>
> Also read the measured `PlanMillis` against the banner's estimate and correct `MACHINE_FACTOR` in
> the parameters cell.

**Send `to_claude/stageB.zip`** — the substrate decision and the visit count come back from it.

In [ ]:
bundle("stageB", [RESULTS / "B", LOGS / "B.log", LOGS / "B_analysis.log", BASE / "dev",
                  LOGS / "A0_profile.log"])

---
## Stage C — is there still a signal?  *(~0.7 h, observation only)*

48 logged games per budget × 3 budgets = 144 games of plain EMCTS on CAPITALS with a 50-tick cap,
mutation-level telemetry on. This is the premise check: does a within-turn table still predict payoff
on the corrected tree?

In [ ]:
# args: threads   budget levels   games/budget   opponent   turn cap   output tree
java("core.data.exp.Stage0SignalRunner", f'{WORKERS} {LEVELS} 48 emcts 50 "{STAGE0_BASE}"',
     log="C", check=True)
run(f'"{PY}" analysis/analyze_stage0.py "{STAGE0_BASE}" --outdir "{RESULTS / "C"}"',
    log="C_analysis", check=True)

In [ ]:
print((RESULTS / "C" / "SUMMARY.md").read_text(encoding="utf-8"))

> ### Gate C
> Mean ρ ≥ 0.15 with a CI excluding zero, at 80k, for at least one granularity. **Fails everywhere
> → stop the line**; the premise is gone and D–G are unfundable.

Also read R9 and R10 — payoff versus expansion-target age, and the re-measured duplicate rate.

**Send `to_claude/stageC.zip`** (the raw `mutlog_*.csv.gz` streams stay on the server — they are the
one heavy artifact; specific ones can be requested by name).

In [ ]:
bundle("stageC", [RESULTS / "C", LOGS / "C.log", LOGS / "C_analysis.log",
                  STAGE0_BASE / "games_summary.csv", STAGE0_BASE / "_meta.txt",
                  str(STAGE0_BASE / "turns_*.csv")],
       note="raw mutlog_*.csv.gz stay on the server; ask for specific ones if a question needs them")

---
## Stage D — biased insertion  *(~5.3 h)*

All 11 arms at 80k → 40k → 400k (8/8/6 visits). That order is the schedule, not a listing: leaves are
checkpointed independently, so an interrupted run keeps the main grid and its budget trend rather
than having spent the time on the expensive confirmation first.

In [ ]:
java("core.data.exp.Stage1Runner", f'{WORKERS} "{BASE}" dev D',
     props=f"-Dstage1.machineFactor={MACHINE_FACTOR} -Dstage1.threads.400000={THREADS_400K}",
     log="D", check=True)
run(f'"{PY}" analysis/analyze_stage1.py "{BASE / "dev"}" --outdir "{RESULTS / "D"}"',
    log="D_analysis", check=True)

In [ ]:
print((RESULTS / "D" / "SUMMARY.md").read_text(encoding="utf-8"))

> ### Gate D
> The champion is the best arm whose CI excludes 0 **and** which survives the wall-clock re-test.
> **Write ε\* and the champion down before Stage F.** Then, in the parameters cell: set `CHAMPION`,
> set `CHAMPION_CONFIRMED = True`, and re-run that cell — E and F check it.
>
> If ε\* is materially different from 0.25, change `Stage1Runner.Z0_EPS`, **re-run the `ab`
> calibration suite**, and re-run the floor probe — the arms and their prices have to move together.

**Send `to_claude/stageD.zip`** — the champion call gets checked against the wall-clock re-test on
the other end before E prices it into six new arms.

In [ ]:
bundle("stageD", [RESULTS / "D", LOGS / "D.log", LOGS / "D_analysis.log", BASE / "dev"])

---
## Stage E — table-directed repair  *(~2.6 h)*

Six arms: `Baseline`, the Stage-D champion carried unchanged as the cross-stage anchor, and the four
`Z0_*` zero-eval arms. Check `RepairCandidateEvals` reads **0** for every `Z0_*` arm. That is a
construction check, not a finding: a non-zero value means the mode is not engaged and the run is a
bug.

In [ ]:
assert CHAMPION_CONFIRMED, ("Stage E prices the champion into six new arms - set CHAMPION and "
                            "CHAMPION_CONFIRMED = True in the parameters cell (and re-run it) "
                            "after reading Gate D.")
java("core.data.exp.Stage1Runner", f'{WORKERS} "{BASE}" dev E {CHAMPION}',
     props=f"-Dstage1.machineFactor={MACHINE_FACTOR} -Dstage1.threads.400000={THREADS_400K}",
     log="E", check=True)
run(f'"{PY}" analysis/analyze_stage1.py "{BASE / "dev"}" --outdir "{RESULTS / "E"}"',
    log="E_analysis", check=True)

In [ ]:
print((RESULTS / "E" / "SUMMARY.md").read_text(encoding="utf-8"))

**Send `to_claude/stageE.zip`** — E1/E2 (the D2 falsification) get read there, and the frozen
choices for F come back in writing.

In [ ]:
bundle("stageE", [RESULTS / "E", LOGS / "E.log", LOGS / "E_analysis.log", BASE / "dev"])

---
## Stage F — the holdout  *(~2.2 h, runs once)*

**Freeze the choices in writing first.** The cell refuses to run until `~/ab_runs/frozen_choices.txt`
exists — put ε\*, the champion, and the repair-granularity winner in it, one per line, before the
holdout is touched. The holdout is worth nothing if it is read twice.

In [ ]:
frozen = DATA / "frozen_choices.txt"
assert CHAMPION_CONFIRMED, "confirm the champion in the parameters cell first (see Gate D)"
assert frozen.exists(), (f"write the frozen choices to {frozen} first - epsilon*, the champion, and "
                         "the repair-granularity winner, one per line. That file is the 'in writing' "
                         "the plan requires before the holdout is read.")
print("frozen choices:\n" + frozen.read_text(encoding="utf-8"))
java("core.data.exp.Stage1Runner", f'{WORKERS} "{BASE}" holdout F {CHAMPION}',
     props=f"-Dstage1.machineFactor={MACHINE_FACTOR} -Dstage1.threads.400000={THREADS_400K}",
     log="F", check=True)
run(f'"{PY}" analysis/analyze_stage1.py "{BASE / "holdout"}" --outdir "{RESULTS / "F"}"',
    log="F_analysis", check=True)

In [ ]:
print((RESULTS / "F" / "SUMMARY.md").read_text(encoding="utf-8"))

> ### Gate F
> A development effect that does not replicate here **is not a result**. Report both numbers side by
> side, always.

**Send `to_claude/stageF.zip`.**

In [ ]:
bundle("stageF", [RESULTS / "F", LOGS / "F.log", LOGS / "F_analysis.log", BASE / "holdout",
                  DATA / "frozen_choices.txt"])

---
## Stage G — the yardstick

Needs work items 9 and 10 (map set 25 → 75, and the matchup list plus the errors-in-variables
mapping), which are not written yet and depend on Stage D's champion — they arrive in a later PR,
with cells added here. Size it from a 4-game timing probe rather than from the plan: full-game turns
are far cheaper than isolation turns, and this line's own bug table records a runtime estimate wrong
by 10× in exactly that direction.

---
## What to keep

Small, and it is what makes a result reproducible rather than merely reported. Everything listed is
already inside the per-stage bundles; this cell is the end-of-line inventory.

In [ ]:
run("git rev-parse HEAD", log="provenance")
for p in [CALIB / "compute_profiles.csv", CALIB / "compute_weights.properties",
          CALIB / "thread_calibration.csv", CALIB / "action_bias_floor.csv",
          PORT / "portfolio.csv", PORT / "_split_dev.txt", PORT / "_split_holdout.txt",
          DATA / "frozen_choices.txt"]:
    print(("  ok  " if pathlib.Path(p).exists() else "  --  ") + str(p))
print(f"\n{LOGS} holds every run's console output; {RESULTS} holds the tables and summaries;")
print(f"{TO_CLAUDE} holds the per-stage zips already sent.")